The program can be used to test different embeddings of text, metrics, and LLM responses

## A compact version of RAG test for deployment in HF

In [204]:
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
import json
import os
import requests
import gradio as gr
import time
import textwrap
import pandas as pd
import tiktoken
import bm25s 
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr
from ordered_set import OrderedSet

import RAG_lib as rlib


In [205]:
if 1:
    # reload rlib if it was changed
    from importlib import reload
    reload(rlib);

## General variables and default options

In [206]:
# Variables
var_dict = {}
# ~ 1000 words or tokens (~ 2 pages) is a good chunk size for scientific texts
# ~ 500 words (~ 1 page) may be OK too
var_dict['chunk_size'] = 1024 # 512
# overlap between chunks - useful if context is at the boundary of a chunk
var_dict['chunk_overlap'] = 200
var_dict['chunk_overlap_BM25'] = 0 # overlap for BM25 RAG
#chunk_overlap_dict = {'BM25':0} # custom overlap for different RAGs
var_dict['RAG_ktop'] =  5 # if 0, then RAG is not used

# BM25, vector
#RAG_representation = 'tokens' # 'tokens', 'vectors'
var_dict['RAG'] = 'words'
#RAG_metric = 'overlap_weighted'
RAG_options = {}
RAG_options['tokens'] = {}
RAG_options['vectors'] = {}

# use a dictionary to query metrics for calculating similarity of embeddings
metric_dict = {'cosine': lambda xs, ys: cosine_similarity(xs, ys)[0]}

# an example of a vector model
RAG_options['vectors']['ollama:nomic-embed-text_cos']  = {}
RAG_options['vectors']['ollama:nomic-embed-text_cos']['embedding']  = \
    OllamaEmbeddings(model="nomic-embed-text").embed_documents
RAG_options['vectors']['ollama:nomic-embed-text_cos']['metric'] = 'cosine'
    

# an example of a token model: words
RAG_options['tokens']['words'] = {}
RAG_options['tokens']['words']['embedding'] = \
    rlib.token_freq_df_embed_documents(tokenizer=rlib.get_words)
RAG_options['tokens']['words']['metric'] = 'token_similarity_weighted'

# an example of a token model: words + BM25
RAG_options['tokens']['BM25'] = {}
RAG_options['tokens']['BM25']['embedding'] = rlib.bm25_embedding
RAG_options['tokens']['BM25']['metric'] = 'BM25_default'

metric_dict['BM25_default'] = rlib.bm25_metric

RAG_dict = {'vectors':['ollama:nomic-embed-text_cos'],
             'tokens':['words', 'BM25']
             }

var_dict['RAG_dict'] = RAG_dict

var_dict['all_RAGs'] = OrderedSet()
for RAGs in RAG_dict.values():
    var_dict['all_RAGs'] = var_dict['all_RAGs'].union(RAGs)

# LLM options
#LLMs = ['gpt-5-mini', 'gpt-5-nano', 'llama3.2', "claude-sonnet-4-5", "gemini-2.5-flash"]
LLM_dict = {'openai':['gpt-5-mini', 'gpt-5-nano'],
             'anthropic':["claude-sonnet-4-5"],
             'google':["gemini-2.5-flash"],
             'hf':[]}


var_dict['LLM_dict'] = LLM_dict

var_dict['all_LLMs'] = OrderedSet()
for LLMs in LLM_dict.values():
    var_dict['all_LLMs'] = var_dict['all_LLMs'].union(LLMs)

#hf_models = ["meta-llama/Llama-3.2-3B:auto", "openai/gpt-oss-20b:auto", "Qwen/Qwen3.5-9B:auto"]

var_dict['RAG'] = 'words'
var_dict['LLM'] = 'gpt-5-mini'

In [222]:
def get_type(model, type_dict):
    for key, models in type_dict.items():
        if model in models:
            return key
    return None

## Meta data, e.g., cost information

## Environment variables

In [223]:
# load environment variables from .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
hf_api_key = os.getenv('HF_TOKEN')

if 1:
    # test if api keys exist
    if openai_api_key:
        print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    else:
        print("OpenAI API Key not set")
        
    if anthropic_api_key:
        print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
    else:
        print("Anthropic API Key not set (and this is optional)")
        
    if google_api_key:
        print(f"Google API Key exists and begins {google_api_key[:2]}")
    else:
        print("Google API Key not set (and this is optional)")

    if hf_api_key:
        print(f"HF API Key exists and begins {hf_api_key[:2]}")
    else:
        print("HF API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
HF API Key exists and begins hf


In [224]:
def openai_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI()
    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response_dict = {'text': response.choices[0].message.content}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        if full_response:
            return response
        else:
            return response_dict
    return chatbot

def google_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI(api_key=google_api_key, 
                 base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response_dict = {'text': response.choices[0].message.content}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot


def anthropic_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = Anthropic()
    def chatbot(messages, full_response=full_response):
        response = bot.messages.create(model=model, messages=messages, max_tokens=1000)
        response_dict = {'text': response.content[0].text}
        usage = response.usage
        # placeholder for usage for anthropic
        response_dict['usage'] = {'prompt_tokens': usage.input_tokens,
                                  'completion_tokens': usage.output_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'output_tokens_details') and usage.output_tokens_details.thinking_tokens:
            reasoning_tokens = usage.output_tokens_details.thinking_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot



def hf_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=hf_api_key
        )

    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response = bot.chat.completions.create(
                model=model,  # ":any other providers" is required
                messages=messages)
        response_text = str(response.choices[0].message.content)
        response_dict = {'text': response_text}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot


def get_bot(var_dict):
    model = var_dict['LLM']
    if model in LLM_dict.get('openai', []):
        return openai_bot(var_dict)
    elif model in LLM_dict.get('anthropic', []):
        return anthropic_bot(var_dict)
    elif model in LLM_dict.get('google', []):
        return google_bot(var_dict)
    elif model in LLM_dict.get('hf', []):
        return hf_bot(var_dict)
    else:
        return None


## RAG preprocessing

In [225]:
# converte pdf to text and save (if not yet converted)
# generalize later for on-demand additions to RAG
fn0 = 'LAT_FBs-1407.7905v1' # Fermi LAT paper about the Fermi bubbles
pdf_fn = 'RAG_data/%s.pdf' % fn0
txt_fn = 'RAG_data/%s.txt' % fn0
rewrite_txt = False
if not os.path.exists(txt_fn) or rewrite_txt:
    text = rlib.pdf2text(pdf_fn)
    with open(txt_fn, "w", encoding="utf-8") as f:
        f.write(text)
else:
    text = open(txt_fn, "r", encoding="utf-8").read()

# calculate the word frequency in the corpus
text_dfs = RAG_options['tokens']['words']['embedding']([text])
word_freq_df = rlib.add_dfs(text_dfs, col_name='frequency')
    

# use the word frequencies as inverse weights
metric_dict['token_similarity_weighted'] = \
    rlib.token_similarity_weighted(word_freq_df)


## Example questions

In [226]:
qfn = 'RAG_data/FB_example_questions.txt'
qs = open(qfn, 'r').readlines()
qs = [q.strip() for q in qs if not q.startswith('#')]

In [227]:

test = 1
if test:
    model_name = 'claude-sonnet-4-5'
    question = 'How many prime numbers are there between 1 and 100?'
    test_messages = [{"role": "user", "content": question}]


In [229]:
model = 'claude-sonnet-4-5'
bot = get_bot(var_dict)
response = bot(messages=test_messages)
print(response)

{'text': 'There are 25 prime numbers between 1 and 100. They are:\n\n2, 3, 5, 7, 11, 13, 17, 19, 23, 29,  \n31, 37, 41, 43, 47, 53, 59, 61, 67, 71,  \n73, 79, 83, 89, 97', 'usage': {'prompt_tokens': 19, 'completion_tokens': 162, 'reasoning_tokens': 64}}


In [230]:
model = 'gpt-5-mini'
bot = get_bot(var_dict)
response = bot(messages=test_messages)
print(response)

{'text': 'There are 25 prime numbers between 1 and 100. They are:\n\n2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97.', 'usage': {'prompt_tokens': 19, 'completion_tokens': 163, 'reasoning_tokens': 64}}


In [231]:
model = 'gemini-2.5-flash'
bot = get_bot(var_dict)
response = bot(messages=test_messages)
print(response)

{'text': 'There are 25 prime numbers between 1 and 100. They are:\n\n2, 3, 5, 7, 11, 13, 17, 19, 23, 29,  \n31, 37, 41, 43, 47, 53, 59, 61, 67, 71,  \n73, 79, 83, 89, 97.', 'usage': {'prompt_tokens': 19, 'completion_tokens': 163, 'reasoning_tokens': 64}}


In [232]:
qs[0:3]

['What are possible origin scenarios for the Fermi bubbles?',
 'What is the distance to the Fermi bubbles in kpc, assuming that they are above the Galactic center?',
 'What is the size of the Fermi bubbles in kpc, assuming that they are above the Galactic center?']

In [233]:
#q = 'What is the capital of France?'
#q = 'How many prime numbers are there between 1 and 100?'
q = qs[2]

test_messages = [{"role": "user", "content": q}]
LLM_dict['hf'] = ["meta-llama/Llama-3.2-1B-Instruct:featherless-ai", # 0
                    "meta-llama/Llama-3.2-3B-Instruct:featherless-ai", # 1
                    "Qwen/Qwen3-0.6B:featherless-ai", # 2
                    "google/gemma-3-4b-it:deepinfra", # 3
                    "Qwen/Qwen3.5-9B:together", # 4
                    "openai/gpt-oss-20b:groq", # 5
                    ]

var_dict['all_LLMs'] = var_dict['all_LLMs'].union(LLM_dict['hf'])

model = LLM_dict['hf'][5]
print(q)
print(model)




What is the size of the Fermi bubbles in kpc, assuming that they are above the Galactic center?
openai/gpt-oss-20b:groq


In [235]:
bot = get_bot(var_dict)
response = bot(messages=test_messages)
print('Answer:', response['text'])
print('Usage:', response['usage'])

Answer: Take the bubbles to be centered on the Galactic center at R0 ≈ 8.3 kpc from the Sun. If they extend to latitude b ≈ 50° (typical quoted value), the vertical height z is

z = R0 · tan b ≈ 8.3 kpc · tan(50°) ≈ 10 kpc.

So each bubble reaches ≈10 kpc above/below the plane (north and south), i.e. a total north–south extent ≈20 kpc. The transverse (longitude) extent is ~40° across, which corresponds to ≈6 kpc wide (≈3 kpc to either side of the GC).

(Using b between ~45°–55° and R0 = 8–8.5 kpc gives heights in the ≈8–12 kpc range.)
Usage: {'prompt_tokens': 28, 'completion_tokens': 955, 'reasoning_tokens': 768}


### Test RAGs with different questions

In [236]:
def RAG_response(query, var_dict):
    chunks = var_dict['chunks']
    RAG = var_dict['RAG']
    ktop = var_dict['RAG_ktop']
    chunk_overlap = var_dict['chunk_overlap']
    if RAG == 'BM25':
        chunk_overlap = var_dict['chunk_overlap_BM25']

    RAG_type = get_type(RAG, RAG_dict)
        
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_text(text)

    embedding = RAG_options[RAG_type][RAG]['embedding']
    metric_name = RAG_options[RAG_type][RAG]['metric']
    metric = metric_dict[metric_name]
    emb_chunks = embedding(chunks)
    emb_qs = embedding([query])

    scores = metric(emb_qs, emb_chunks)
    inds = np.argsort(scores)[::-1][:ktop]

    message = ''
    for ind in inds:
        message += '%s\n\n' % chunks[ind]
    return message



    


In [243]:
var_dict['RAG']

'words'

In [248]:
#RAG = 'ollama:nomic-embed-text_cos'
#RAG_type = get_type(RAG, RAG_dict)
chunk_overlap = var_dict['chunk_overlap']
if var_dict['RAG'] == 'BM25':
    chunk_overlap = var_dict['chunk_overlap_BM25']


## Split text in chunks

In [249]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_text(text)
var_dict['chunks'] = chunks
print('The number of chunks is', len(chunks))

The number of chunks is 186


In [250]:
q = qs[1]
print(RAG_response(q, var_dict))

the electron spectrum by a power law with an exponential cutoff ∝E−ne−E/E cut. The best fit parameters
aren= 2.17±0.05[stat]+0.33
−0.89[syst] and Ecut= 1.25±0.13[stat]+1.73
−0.68[syst] TeV. The corresponding IC
spectra are shown in Figure 34 on the left. The details of the calculation can be found in Appendix
B.1. The indices and the cutoff values for different foreground models and definitions of the templates
of Loop I and the bubbles are shown in Figure 34 on the right. The bremsstrahlung emission is at least
two orders of magnitude smaller than the IC emission for a characteristic gas density nH≲0.01 cm−3at
a few kpc from the Galactic plane (Snowden et al. 1997), and can be neglected.
We will assume that the center of the bubbles is at b= 25◦, i.e., the distance to the center of
the bubbles is R=R⊙/cosb= 9.4 kpc, where R⊙= 8.5 kpc is the distance to the GC. The total
energy contained in the electron population inside the bubbles above 1 GeV is (1 .0±0.2[stat]+6.0
−1.0[syst]) ×

201

In [ ]:
var_dict['all_LLMs']

OrderedSet(['gpt-5-mini', 'gpt-5-nano', 'claude-sonnet-4-5', 'gemini-2.5-flash'])

In [ ]:
def list2lines(lst):
    res = ''
    for x in lst:
        res += f'\t- {x}\n'
    return res

def dict2lines(dct):
    res = ''
    for key, value in dct.items():
        res += f'{key}:'
        if type(value) is list:
            res += '\n' + list2lines(value)
        else:
            res += f' {value}'
    return res

commands_dict = {
    '/help': 'print the list of available commands etc.',
    '/llm': 'print current LLM',
    '/llms': 'print the available LLMs',
    '/llm=LLM': 'change LLM specified by "LLM"',
}
special_commands = ['/help']

def handle_model_commands(message, var_dict={}):
    if not message.startswith('/'):
        return None
    command = message.split(' ')[0]
    
    if command in special_commands:
        return handle_special_commands(command)
    elif message.find('=') > -1:
        var_name = message.split('=')[0]
    else:
        var_name = message.split(' ')[0]
    var_name = var_name.replace('/', '').strip()

    if var_name not in var_dict.keys():
        out_message = f'{var_name} is not found in variables dictionary.\n'
        out_message += f'Available variables: {list(var_dict.keys())}'
        return out_message
    
    if message.find('=') > -1:
        new_value = message.split('=')[1].strip()
        old_value = var_dict[var_name]
        
        if var_name == 'LLM' and new_value not in var_dict['all_LLMs']:
            out_message = f'Model {new_value} not found.\n'
            out_message += f'Available models: {var_dict[f'all_LLMs']}'
            return out_message
        elif var_name == 'RAG' and new_value not in var_dict['all_RAGs']:
            out_message = f'Model {new_value} not found.\n'
            out_message += f'Available models: {var_dict[f'all_RAGs']}'
            return out_message
        if new_value.isnumeric():
            new_value = int(new_value)

        var_dict[var_name] = new_value

        out_message = f'Old {var_name} value: {old_value}, new {var_name} value: {new_value}'
    else:
        out_message = f'Current {var_name} = {var_dict[var_name]}'

    return out_message


def handle_help(message, var_dict={}):
    return 'Print information about commands and other options'
    
def execute(message, var_dict=var_dict):
    if not message.startswith('/'):
        return None
    res = handle_model_commands(message, var_dict=var_dict)
    if res: return res
    res = handle_help(message, var_dict=var_dict)
    if res: return res
    return None


In [308]:
message = ' /wer = asdf'
command = message.split('=')
message

' /wer = asdf'

In [309]:
def chat(message, history, var_dict=var_dict):
    messages = history + [message]
    if message.startswith('/'):
        response = execute(message, var_dict=var_dict)
    elif message.startswith('@'):
        if message.lower().startswith('@rag'):
            query = message.replace('@rag', '').strip()
            response = RAG_response(query, var_dict)
        elif message.lower().startswith('@llm'):
            query = message.replace('@llm', '').strip()
            messages = [{"role": "user", "content": query}]
            if var_dict['RAG_ktop'] > 0:
                RAG_context = RAG_response(query, var_dict)
                messages.append({"role": "user", "content": RAG_context})
            bot = get_bot(var_dict)
            response = bot(messages)
    else:
        response = message + ' echo'
    return response

In [310]:
chat_box = gr.ChatInterface(chat, type="messages",     
                            textbox=gr.Textbox(placeholder="", lines=1, max_lines=30))
chat_box.launch()

* Running on local URL:  http://127.0.0.1:7910
* To create a public link, set `share=True` in `launch()`.
